# Imports

In [11]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 11 16:05:15 2023

@author: andrej
"""
import sys,os
sys.path.append(r'C:/data/EnergyTrading/Python/')

import pandas as pd
from Database.TPData import TPData
from Database.DB_reader import Database
from datetime import date, timedelta, datetime, time

import cx_Oracle
try:
    cx_Oracle.init_oracle_client(lib_dir=r"C:\Users\user\Downloads\instantclient_21_11")
except:
    pass

from OrderBook.OrderBook import OrderBookSnaps
import os
import itertools

In [12]:
def previous_working_day(d):
    if d.weekday() == 0:  # Monday
        return d - timedelta(days=3)  # Previous Friday
    elif d.weekday() == 6:  # Sunday
        return d - timedelta(days=2)  # Previous Friday
    else:
        return d - timedelta(days=1)  # Previous day

In [ ]:
local_db_config_path=''

# Email utilities

In [13]:
from Utilities.email_sending import send_plain_email, send_html_email

EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = "zubal_andrej@energytrading.sk" # can also be a list of recipients

# Defining date range

In [14]:
mkt_list = ['de']
tenor_list = ['m', 'q', 'y']
prod = 'base'
venue_list = ['eex']
tn_dict = {'m': [1, 2, 3, 4],
           'q': [1, 2, 3, 4],
           'y': [1, 2]}

# tenor_list = ['m']
# prod = 'base'
# venue_list = ['eex']
# tn_dict = {'m': [1]}


prev_wday = previous_working_day(datetime.today().date())
start_date = datetime.combine(prev_wday, time(0, 0))
end_date = datetime.combine(prev_wday, time(0, 0))
del prev_wday

n_s = 2
dates = pd.date_range(start_date, end_date, freq='B')

combinations = []
for mkt in mkt_list:
    for tenor in tenor_list:
        for tn in tn_dict[tenor]:
            combinations.append((mkt, tenor, tn))

mkt_list_ = [x[0] for x in combinations]
tenor_list_ = [x[1] for x in combinations]
tn_list_ = [x[2] for x in combinations]

product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list_, tn_list_)]

for m, t, prod_d, tn in zip(mkt_list_, tenor_list_, product_date, tn_list_):
    for dt, p_d in zip(dates, prod_d):
        print((m, t, prod, venue_list, p_d, dt))
        try:
            data_class = TPDataDa()
            ob_class = OrderBookSnaps(verbose=True)
            
            bT=dt.to_pydatetime().replace(hour=8, minute=0, second=0)
            eT=timestamp.to_pydatetime().replace(hour=18, minute=0, second=0)
            
            LoB_dict, time_list = load_ob(m, t, bT, p_d, bT, eT)
            ob_class.update_data(LoB_dict, time_list)
            
            xx = pd.concat([ob_class.best_bid_all, ob_class.best_ask_all], axis=1)
            xx.columns = ['bidbestprice', 'askbestprice']
            xx1 = data_class.process_best_orders(xx)
            
            xx1=xx1.reset_index()
            xx1['relativecontract']=m+'_'+t+'_'+str(tn)
            xx1['upload_time']= datetime.now()
            xx1=xx1[['index','relativecontract', 'bidbestprice', 'askbestprice', 'upload_time']]
            
           
            
        except:
            print('Failed')


('de', 'm', 'base', ['eex'], Timestamp('2024-08-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'm', 'base', ['eex'], Timestamp('2024-09-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'm', 'base', ['eex'], Timestamp('2024-10-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'm', 'base', ['eex'], Timestamp('2024-11-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'q', 'base', ['eex'], Timestamp('2024-10-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'q', 'base', ['eex'], Timestamp('2025-01-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'q', 'base', ['eex'], Timestamp('2025-04-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'q', 'base', ['eex'], Timestamp('2025-07-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'y', 'base', ['eex'], Timestamp('2025-01-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))
('de', 'y', 'base', ['eex'], Timestamp('2026-01-01 00:00:00'), Timestamp('2024-06-27 00:00:00'))


# Loading data and sending emails

In [4]:
local_db_config_path=r'C:\data\EnergyTrading\configDB.json'

try:
    for date in date_range:
        
        conn = Database('OracleSQL',path_name=local_db_config_path)

        query=f"""select * from  rove_od.trayport_vw_trades 
        WHERE 1=1
        and to_date(datetime)>=to_date('{date}', 'YYYY-MM-DD')
        and to_date(datetime)<=to_date('{date}', 'YYYY-MM-DD')""" # where rownum <= 100"""
        #query="""select * from  public.trayport_orders limit 100"""

        df1=conn.execute(query)

        conn = Database(path_name=local_db_config_path)
        conn._connect()
        df1.to_sql('trayport_vw_trades', conn.engine, schema='public', if_exists='append', index=False)

        print('\n')
        print(f'{date} number of records: {df1.shape[0]}')

        #email sending in case of success, also sending number of records uploaded
        send_plain_email(
          RECIPIENT, 
        "SUCCESS: trade_data_daily_update_remote_comp job", f'{date} number of records: {df1.shape[0]}',
        email_password=EMAIL_PASSWORD
    )
        print('\n')

        conn._disconnect()

except:
    send_plain_email(
          RECIPIENT, 
        "FAIL: trade_data_daily_update_remote_comp", f'The upload of data failed for this run, please check what is the issue.',
        email_password=EMAIL_PASSWORD
    )